# Web Scraping + Regressão Linear para previsão de preço de imóveis

> Neste web scraper podemos verificar preço de imóveis de Brasília, especialmente no Plano Piloto.

**Etapa 1 – Coleta de Dados (Web Scraping).** Coletar dados reais de um site de imóveis (VivaReal) e estruturá-los em um DataFrame pandas (Objetivos de Aprendizagem 1 e 2).

| Requisito mínimo da coleta (PDF) | Como é atendido |
|---|---|
| Mínimo de 100 imóveis (ideal: 300+) | até 300 anúncios (`MAX_IMOVEIS`) |
| Filtrar por uma cidade específica | Brasília: Plano Piloto e Lagos |
| `preco`, `area_m2`, `quartos`, `banheiros`, `vagas`, `bairro`, `tipo` | uma coluna para cada, em `imoveis_brutos.csv` |

A saída ainda é **texto cru** (`"R$ 1.200.000"`, `"75 m²"`); a conversão é tarefa da Etapa 2.

## Imports

`requests` e `BeautifulSoup` são as bibliotecas indicadas no PDF. O `curl_cffi` tem a mesma API do `requests` e o substitui na coleta. O motivo aparece no Recon 3.

In [1]:
import os
import random
import re
import time

import pandas as pd
import requests                          # usado só para mostrar o bloqueio (403)
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi   # requests com handshake TLS de navegador

## Setando variáveis

O PDF pede uma **cidade específica** para reduzir a variabilidade geográfica. Buscar só "Brasília" traz o DF inteiro, então a busca é feita por bairro, pela URL `.../bairros/<bairro>/<tipo>/`. A cota por bairro evita que a amostra fique concentrada em um só lugar, o que conta como viés de *seleção geográfica*, uma das limitações que o relatório deve discutir.

In [2]:
url = "https://www.vivareal.com.br/venda/distrito-federal/brasilia/"

# nome que vai para a coluna "bairro" -> slug usado na URL
BAIRROS = {
    "Asa Sul": "asa-sul",
    "Asa Norte": "asa-norte",
    "Noroeste": "setor-noroeste",
    "Sudoeste": "setor-sudoeste",
    "Lago Sul": "lago-sul",
    "Lago Norte": "lago-norte",
}
# apartamento vem antes: com casa primeiro, os lotes grandes dominam a amostra
TIPOS = {"apartamento": "apartamento_residencial", "casa": "casa_residencial"}

MAX_IMOVEIS = 300
POR_BAIRRO = MAX_IMOVEIS // len(BAIRROS)   # 300 // 6 = 50 anúncios por bairro
COLETAR = not os.path.exists("imoveis_brutos.csv")   # só raspa sem CSV salvo; use True para coletar de novo

# prints para depuração
print(f"Bairros: {list(BAIRROS)}")
print(f"Tipos: {list(TIPOS)}")
print(f"Máximo: {MAX_IMOVEIS} imóveis ({POR_BAIRRO} por bairro)")
print(f"Coletar agora? {COLETAR}\n")

Bairros: ['Asa Sul', 'Asa Norte', 'Noroeste', 'Sudoeste', 'Lago Sul', 'Lago Norte']
Tipos: ['apartamento', 'casa']
Máximo: 300 imóveis (50 por bairro)
Coletar agora? False



## Recon 1: escopo

Antes de testar qualquer alvo, vou definir o escopo. Aqui nesta depuração podemos ver o `robots.txt`: só usamos URLs permitidas (sem `onde=`, `tipos=`, `quartos=`...), apenas com `?pagina=N`.

Respeitar o `robots.txt` faz parte das **considerações éticas sobre scraping** (Etapa 5, item 7).

In [3]:
robots = cffi.get("https://www.vivareal.com.br/robots.txt", impersonate="chrome", timeout=20).text
for linha in robots.splitlines()[:12]:   # só o começo: as regras para "User-agent: *"
    print(linha)

User-agent: *
Allow: /
Disallow: /inmueble/
Disallow: /preview/
Disallow: /parceiros/
Disallow: /projects/
Disallow: /ajax/pageserver/
Disallow: /buy/
Disallow: /rent/
Disallow: /*placeId=*
Disallow: /*placeLabel=*
Disallow: /*listingUse=*


## Recon 2: tem WAF na frente?

O PDF classifica o VivaReal como *"HTML dinâmico (requer Selenium/Playwright)"*. Na prática, os anúncios já vêm no HTML da resposta. O obstáculo é outro: um **WAF** (*Web Application Firewall*), que os headers da resposta entregam.

In [4]:
resposta = cffi.get(url, impersonate="chrome", timeout=20)
print("server:", resposta.headers.get("server"))    # "cloudflare" = WAF na frente do site
print("cf-ray:", resposta.headers.get("cf-ray"))    # id da requisição dentro do Cloudflare
print("cookies:", list(resposta.cookies.keys()))    # __cf_bm = cookie de gerenciamento de bots

server: cloudflare
cf-ray: a437bc8e0816932c-EWR
cookies: ['z_user_id', 'z_exp', '__cf_bm', '_cfuvid']


## Recon 3: a digital TLS

O `robots.txt` não bloqueia ninguém. Quem bloqueia é o **Cloudflare**, e ele identifica o cliente pelo **handshake TLS** (hash JA3/JA4) e pela versão do HTTP, antes mesmo de ler o `User-Agent`. O `tls.peet.ws` devolve a digital que o servidor enxerga:

In [5]:
eco = "https://tls.peet.ws/api/clean"   # serviço que devolve a digital TLS do cliente
digital_requests = requests.get(eco, timeout=20).json()
digital_chrome = cffi.get(eco, impersonate="chrome", timeout=20).json()

# ja4 = resumo do handshake TLS | http_version: requests só fala HTTP/1.1, o Chrome usa h2
print("requests  ->", digital_requests["ja4"], digital_requests["http_version"])
print("curl_cffi ->", digital_chrome["ja4"], digital_chrome["http_version"])

requests  -> t13d1713h1_ab0a1bf427ad_8537cf56674e HTTP/1.1
curl_cffi -> t13d1516h2_8daaf6152771_806a8c22fdea h2


## Bypass: `requests` × `curl_cffi`

Com o `requests`, mesmo com `User-Agent` de Chrome, o JA4 continua dizendo "Python", e o resultado é 403. O `curl_cffi` (`impersonate="chrome"`) reproduz o handshake do Chrome e passa. Com ele **não** se troca o `User-Agent`: a versão precisa bater com a digital TLS.

Não há CAPTCHA resolvido nem desafio JavaScript contornado: só a conexão passa a se parecer com a de um navegador.

In [ ]:
headers = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"),
    "Accept-Language": "pt-BR,pt;q=0.9",
}

teste = url + "apartamento_residencial/"

resposta = requests.get(teste, params={"pagina": 1}, headers=headers, timeout=20)
print(f"[requests]  Status: {resposta.status_code}")   # 403: bloqueado pelo WAF
print(f"URL: {resposta.url}")
print(f"Tamanho do body: {len(resposta.text)}")        # 5 mil caracteres = página de bloqueio

resposta = cffi.get(teste, params={"pagina": 1}, impersonate="chrome", timeout=20)
print(f"[curl_cffi] Status: {resposta.status_code}")   # 200
print(f"URL: {resposta.url}")
print(f"Tamanho do body: {len(resposta.text)}")        # 1,3 milhão = página com os anúncios

[requests]  Status: 403
URL: https://www.vivareal.com.br/venda/distrito-federal/brasilia/apartamento_residencial/?pagina=1
Tamanho do body: 5487
[curl_cffi] Status: 200
URL: https://www.vivareal.com.br/venda/distrito-federal/brasilia/apartamento_residencial/?pagina=1
Tamanho do body: 1326327


## Extraindo os dados de um card

Cada anúncio é um `<li data-cy="rp-property-cd">`. As variáveis do PDF saem destes seletores:

| Variável | Seletor `data-cy` |
|---|---|
| `preco` | `rp-cardProperty-price-txt` (1º `<p>`) |
| `area_m2` | `rp-cardProperty-propertyArea-txt` |
| `quartos` / `banheiros` / `vagas` | `rp-cardProperty-bedroomQuantity-txt` / `...bathroomQuantity-txt` / `...parkingSpacesQuantity-txt` |
| `bairro` e `tipo` | vêm da URL da busca (o texto do card é inconsistente: "Norte", "Setor de Habitações...") |

O `.sr-only` é texto escondido para leitores de tela ("Tamanho do imóvel") e é removido. Nome e telefone do anunciante **não** são coletados (LGPD).

In [ ]:
def texto(card, data_cy):
    """Texto visível do elemento com esse data-cy, ou None se ele não existir/estiver vazio."""
    elemento = card.select_one(f'[data-cy="{data_cy}"]')
    if elemento is None:                     # ex.: anúncio sem vaga informada
        return None
    for escondido in elemento.select(".sr-only"):
        escondido.decompose()                # tira o rótulo de acessibilidade
    return elemento.get_text(" ", strip=True) or None


def extrair_card(card, bairro, tipo):
    """Um card -> um dicionário (uma linha da tabela), ainda como texto."""
    link = card.select_one("a[href]")
    achado = re.search(r"-id-(\d+)", link["href"]) if link else None   # ex.: /imovel/apartamento-3-quartos-asa-sul-id-12345678
    preco = card.select_one('[data-cy="rp-cardProperty-price-txt"] p')   # 1º <p> = preço; o 2º é cond./IPTU
    return {
        "id": achado.group(1) if achado else None,
        "preco": preco.get_text(" ", strip=True) if preco else None,    # ex.: "R$ 1.200.000"
        "area_m2": texto(card, "rp-cardProperty-propertyArea-txt"),
        "quartos": texto(card, "rp-cardProperty-bedroomQuantity-txt"),
        "banheiros": texto(card, "rp-cardProperty-bathroomQuantity-txt"),   
        "vagas": texto(card, "rp-cardProperty-parkingSpacesQuantity-txt"),
        "bairro": bairro,
        "tipo": tipo,
    }

## Coleta (até 300 imóveis)

Uma sessão só (mantém o cookie `__cf_bm` como um navegador), pausa aleatória (*jitter*) entre as páginas para não cair na regra de *rate limit* e, se vier 403/429, espera e tenta de novo.

A pausa de 2 a 4 s também é uma questão ética: não sobrecarregar o servidor. O uso é acadêmico e a base não é redistribuída.

In [8]:
def baixar(sessao, url_busca, pagina, tentativas=3):
    """Lista de cards de uma página de busca ([] se não existir ou se o bloqueio persistir)."""
    for tentativa in range(tentativas):
        resposta = sessao.get(url_busca, params={"pagina": pagina}, timeout=30)
        if resposta.status_code in (403, 429):   # WAF/rate limit: espera 15 s, 30 s, 45 s
            print(f"   {resposta.status_code}, esperando para tentar de novo...")
            time.sleep(15 * (tentativa + 1))
            continue
        if resposta.status_code != 200:          # 404 = não há esse tipo no bairro
            return []
        return BeautifulSoup(resposta.text, "html.parser").select('[data-cy="rp-property-cd"]')
    return []


def coletar():
    """Percorre bairro × tipo × página até completar POR_BAIRRO anúncios únicos em cada bairro."""
    sessao = cffi.Session(impersonate="chrome")
    imoveis, vistos = [], set()

    for bairro, slug_bairro in BAIRROS.items():
        no_bairro = 0
        for tipo, slug_tipo in TIPOS.items():
            url_busca = f"{url}bairros/{slug_bairro}/{slug_tipo}/"
            for pagina in range(1, 11):                    # teto de 10 páginas por busca
                if no_bairro >= POR_BAIRRO:
                    break
                cards = baixar(sessao, url_busca, pagina)
                print(f"{bairro} | {tipo} | página {pagina}: {len(cards)} cards")
                time.sleep(random.uniform(2, 4))           # jitter
                if not cards:                              # página vazia = fim da busca
                    break
                for card in cards:
                    imovel = extrair_card(card, bairro, tipo)
                    if imovel["id"] is None or imovel["id"] in vistos:
                        continue                           # card agrupado pelo site (href="#") ou "Destaque" repetido
                    vistos.add(imovel["id"])
                    imoveis.append(imovel)
                    no_bairro += 1
                    if no_bairro >= POR_BAIRRO:
                        break
        print(f"-> {bairro}: {no_bairro} imóveis\n")

    print(f"--- Total coletado: {len(imoveis)} imóveis únicos ---\n")
    return pd.DataFrame(imoveis)                           # lista de dicionários -> tabela


if COLETAR:
    bruto = coletar()
    bruto.to_csv("imoveis_brutos.csv", index=False, encoding="utf-8-sig")   # utf-8-sig: Excel abre os acentos certos

bruto = pd.read_csv("imoveis_brutos.csv", dtype=str)   # dtype=str: mantém tudo como texto cru
print(f"CSV carregado: {bruto.shape[0]} linhas x {bruto.shape[1]} colunas")

CSV carregado: 299 linhas x 14 colunas


## Conferindo

Aqui os dados já estão **estruturados em um DataFrame pandas** (Objetivo 2): uma linha por imóvel, uma coluna por variável pedida no PDF. Os valores ausentes (`NaN`) e as faixas como `"2 - 3"` são tratados na Etapa 2.

OBS.: Esta parte é ignorável, é mais para depuração do meu código.

In [10]:
print("--- Imóveis por bairro ---")
print(bruto["bairro"].value_counts(), "\n")

print("--- Imóveis por tipo ---")
print(bruto["tipo"].value_counts(), "\n")

print("--- Valores ausentes (texto cru) ---")
print(bruto.isna().sum(), "\n")

bruto.head(10)

--- Imóveis por bairro ---
bairro
Asa Sul       50
Asa Norte     50
Noroeste      50
Sudoeste      50
Lago Norte    50
Lago Sul      49
Name: count, dtype: int64 

--- Imóveis por tipo ---
tipo
apartamento    225
casa            74
Name: count, dtype: int64 

--- Valores ausentes (texto cru) ---
id              0
preco           0
area_m2         0
quartos         2
banheiros       1
vagas          55
bairro          0
tipo            0
condominio      3
iptu            3
endereco        0
titulo          0
link            0
coletado_em     0
dtype: int64 



,id,preco,area_m2,quartos,banheiros,vagas,bairro,tipo,condominio,iptu,endereco,titulo,link,coletado_em
0,2758842032,R$ 69.000,57 m²,2,2,1,Asa Sul,apartamento,isento,isento,EQS 414/415,"Apartamento com piscina no condomínio, 2 dormi...",https://www.vivareal.com.br/imovel/apartamento...,2026-09-30
1,2889651112,A partir de R$ 1.964.760,110 - 275 m²,3,3 - 5,2 - 4,Asa Sul,apartamento,NaN,NaN,"SQS 305, Asa Sul, Brasília","Apartamento à venda na planta com 3 quartos, s...",https://www.vivareal.com.br/imoveis-lancamento...,2026-09-30
2,2914832537,R$ 80.000,50 m²,2,1,1,Asa Sul,apartamento,isento,R$ 200,EQS 414/415,"Apartamento para comprar com 50 m², 2 quartos,...",https://www.vivareal.com.br/imovel/apartamento...,2026-09-30
3,2761257682,R$ 69.000,57 m²,2,2,1,Asa Sul,apartamento,isento,isento,SQS 414,"Apartamento com elevador no condomínio, 2 dorm...",https://www.vivareal.com.br/imovel/apartamento...,2026-09-30
4,2914838039,R$ 300.000,59 m²,2,1,NaN,Asa Sul,apartamento,R$ 359,isento,EQS 414/415,"Apartamento para comprar com 59 m², 2 quartos,...",https://www.vivareal.com.br/imovel/apartamento...,2026-09-30
5,2898681724,R$ 1.800.000,136 m²,3,3,1,Asa Sul,apartamento,R$ 1.170,R$ 2.630,SQS 109 Bloco E,"Apartamento para comprar com 136 m², 3 quartos...",https://www.vivareal.com.br/imovel/apartamento...,2026-09-30
6,2902478128,R$ 710.000,54 m²,2,1,NaN,Asa Sul,apartamento,R$ 500,isento,SQS 410 Bloco R,"Apartamento para comprar com 54 m², 2 quartos,...",https://www.vivareal.com.br/imovel/apartamento...,2026-09-30
7,2886677717,R$ 500.000,116 m²,3,3,NaN,Asa Sul,apartamento,isento,isento,SBS Quadra 2,"Apartamento mobiliado com varanda, 3 suítes e ...",https://www.vivareal.com.br/imovel/apartamento...,2026-09-30
8,2881428813,R$ 1.150.000,87 m²,3,3,NaN,Asa Sul,apartamento,R$ 1.259,R$ 384,SQS 205 Bloco C,"Apartamento com 3 dormitórios, 1 suíte e 87 m²...",https://www.vivareal.com.br/imovel/apartamento...,2026-09-30
9,2836890236,R$ 650.000,42 m²,1,2,NaN,Asa Sul,apartamento,R$ 675,isento,CLS 205 Bloco B,"Apartamento para comprar com 42 m², 1 quarto, ...",https://www.vivareal.com.br/imovel/apartamento...,2026-09-30
